# Primordial non-Gaussianity in the collapsed fraction and the halo mass function

This notebook shows how a local primordial non-Gaussianity of amplitude $f_{\rm NL}$
changes the halo mass function, and therefore the collapsed fraction that drives
reionization in `21cmFirstCLASS_NG`.

It deliberately **imports the production code** from `py21cmfast.generate_ICs`
rather than re-implementing it. A notebook that carries its own copy of the
integrals will drift away from the code that actually runs, and then it is
documenting something that no longer exists. Everything plotted below is
computed by the same functions the simulation uses.

**What the notebook covers**

1. The three-point functions $\mu_3$ of the smoothed linear density field
2. The reduced skewness $\kappa_3 = \mu_3/\sigma^3$, and a check that it is
   redshift independent
3. The Edgeworth correction to the unconditional mass function
4. How the correction depends on $f_{\rm NL}$, on redshift, and on $k_{\rm cut}$

**References**

* Sabti, Muñoz & Blas, [arXiv:2009.01245](https://arxiv.org/abs/2009.01245) —
  the Edgeworth expansion, Eqs. (38)–(40), and the $\kappa_3$ calculation
* LoVerde, Miller, Shandera & Verde,
  [arXiv:0711.4126](https://arxiv.org/abs/0711.4126) — the original derivation
* Planck 2018 IX, [arXiv:1905.05697](https://arxiv.org/abs/1905.05697) —
  the $f_{\rm NL}$ convention used here

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

# The production implementation. Nothing below reimplements these.
from py21cmfast.generate_ICs import window, der_window, Pphi, curlM, der_curlM, tpf

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

## 1. Setup

We use the Planck 2018 cosmology. `KCUT_FNL` is the lower cut-off $k_{\rm cut}$
applied to every leg of the bispectrum: modes below it stay Gaussian. Setting it
well below the box fundamental mode gives scale-independent non-Gaussianity.

The `tpf` function expects objects with attributes rather than dictionaries,
because that is how `CosmoParams` and `GlobalParams` arrive from the wrapper.
The two small classes below stand in for them.

In [ ]:
class CosmoParams:
    hlittle     = 0.6736
    OMb         = 0.0493
    OMm         = 0.3153
    A_s         = 2.1e-9
    POWER_INDEX = 0.9649
    F_NL        = 100.0     # local non-Gaussianity, CMB (Planck) convention
    KCUT_FNL    = 1e-2      # lower k cut-off on the bispectrum [1/Mpc]

class GlobalParams:
    pass

cosmo_params  = CosmoParams()
global_params = GlobalParams()

# Mass grid. Must match the one used for sigma(M) in the simulation, so that the
# interpolation indices line up on the C side.
log10_M_array = np.linspace(4., 20., 300)
M_array = 10 ** log10_M_array

## 2. The transfer function

We need $T_\zeta(k, 0)$, the transfer function from the primordial curvature
perturbation $\zeta$ to the matter density at $z = 0$.

Two points worth being careful about, because both are easy to get wrong and
neither shows up as an obvious error:

* **The normalisation.** `Pphi` carries a factor $(3/5)^2$ and `curlM` divides
  by $3/5$. These cancel in $\sigma_M$, so that quantity is insensitive to the
  convention. They do **not** cancel in $\kappa_3$, where a factor $3/5$
  survives — and that surviving factor is exactly what makes `F_NL` the
  CMB-convention $f_{\rm NL}$ comparable to Planck's $\sigma(f_{\rm NL}) = 5.1$.
* **`P_k_max_1/Mpc`.** The three-point integrals reach well beyond the range
  $\sigma(M)$ alone needs, so this has to be large.

In [ ]:
CLASS_params = {
    'h':           cosmo_params.hlittle,
    'Omega_b':     cosmo_params.OMb,
    'Omega_cdm':   cosmo_params.OMm - cosmo_params.OMb,
    'n_s':         cosmo_params.POWER_INDEX,
    'A_s':         cosmo_params.A_s,
    'T_cmb':       2.728,
    'output':      'mTk,mPk',
    'z_pk':        0.,
    'P_k_max_1/Mpc': 3000.,
}

CLASS_OUTPUT = Class()
CLASS_OUTPUT.set(CLASS_params)
CLASS_OUTPUT.compute()

print(f"sigma8 = {CLASS_OUTPUT.sigma8():.4f}   (Planck 2018: 0.8111)")

A useful first check: $\sigma_8$ from CLASS against the Planck value. If the
amplitude is wrong here, every number downstream is wrong by the same factor
cubed in $\mu_3$, and $\kappa_3$ would still look plausible because it is a
ratio. The two checks are sensitive to different mistakes, so both are needed.

In [ ]:
def interpolate_transfer(T_CLASS, k_CLASS, k_output):
    '''Interpolate onto k_output, extrapolating as a power law beyond the range.'''
    from scipy.interpolate import interp1d
    f = interp1d(k_CLASS, T_CLASS, kind='cubic', bounds_error=False, fill_value=0.)
    T = f(k_output)
    if max(k_output) > max(k_CLASS):
        i = np.where(k_output > k_CLASS[-1])[0][0] - 1
        if abs(T[i]) > 0 and abs(T[i-1]) > 0:
            slope = np.log(T[i]/T[i-1]) / np.log(k_output[i]/k_output[i-1])
            T[i:] = T[i] * (k_output[i:]/k_output[i])**slope
    if min(k_output) < min(k_CLASS):
        i = np.where(k_output < k_CLASS[0])[0][-1] + 1
        if abs(T[i]) > 0 and abs(T[i+1]) > 0:
            slope = np.log(T[i]/T[i+1]) / np.log(k_output[i]/k_output[i+1])
            T[:i] = T[i] * (k_output[:i]/k_output[i])**slope
    return abs(T)


# The wavenumber grid the simulation tabulates transfer functions on.
LOG_K_ARR = np.linspace(-5.15, 3.05, 144)
k_output = 10 ** LOG_K_ARR

transfer = CLASS_OUTPUT.get_transfer(0., 'camb')
k_CLASS = CLASS_OUTPUT.get_transfer(z=0.)['k (h/Mpc)'] * cosmo_params.hlittle

global_params.LOG_K_ARR_FOR_TRANSFERS = list(LOG_K_ARR)
global_params.T_ZETA_TRANSFER = list(
    interpolate_transfer(transfer['-T_tot/k2'], k_CLASS, k_output))

print(f"k range: {k_output.min():.2e} to {k_output.max():.2e} 1/Mpc")

## 3. The three-point functions

`tpf` returns four tables on the $(M_n, M_m)$ grid: the cumulant
$\langle\delta_n \delta_n \delta_m\rangle$ and its derivative with respect to
$M_n$ in three cases, according to whether $M_n$ is equal to, below, or above
$M_m$. The simulation needs all four because the conditional mass function
couples two scales.

This is the expensive step. It is chunked over the mass axis internally so that
peak memory stays near 1.5 GiB; see `_FM_MASS_CHUNK` in `generate_ICs.py`.

In [ ]:
%time mu3, dmu3_diag, dmu3_lower, dmu3_upper = tpf(log10_M_array, cosmo_params, global_params)

print(f"shapes: {mu3.shape}")
print(f"mu3 on the diagonal spans {np.diag(mu3).min():.3e} to {np.diag(mu3).max():.3e}")

## 4. The reduced skewness

$\kappa_3 = \mu_3 / \sigma^3$ is the quantity the Edgeworth expansion is
organised in. Two things to look for:

* **Magnitude.** For $k_{\rm cut} = 0.1\,\mathrm{Mpc}^{-1}$, Fig. 4 of
  arXiv:2009.01245 gives $\kappa_3 \simeq 3\!-\!4.5 \times 10^{-4} f_{\rm NL}$
  over $10^5$–$10^{11} M_\odot$. A smaller cut-off admits more large-scale
  modes and raises it.
* **Shape.** $\kappa_3$ falls slowly with mass, and drops to zero near the mass
  whose filter radius matches $1/k_{\rm cut}$, namely
  $M_{\rm cut} = 4\pi \rho_m / (3 k_{\rm cut}^3)$. That turnover is the clearest
  signature that the cut-off has been applied correctly.

In [ ]:
rho_crit = 2.7754e11 * cosmo_params.hlittle**2          # Msun / Mpc^3
rho_m = rho_crit * cosmo_params.OMm

sigma_M = np.array([CLASS_OUTPUT.sigma((3*M/(4*np.pi*rho_m))**(1/3.), 0.)
                    for M in M_array])

kappa3 = np.diag(mu3) / sigma_M**3

M_cut = 4*np.pi*rho_m / (3 * cosmo_params.KCUT_FNL**3)
print(f"expected cut-off mass for k_cut = {cosmo_params.KCUT_FNL} /Mpc: {M_cut:.2e} Msun")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogx(M_array, 1e4 * kappa3 / cosmo_params.F_NL)
ax[0].axvline(M_cut, ls=':', color='0.5')
ax[0].text(M_cut, ax[0].get_ylim()[1]*0.9, '  $M_{\\rm cut}$', color='0.4')
ax[0].set(xlabel=r'$M\ [M_\odot]$', ylabel=r'$10^4\,\kappa_3/f_{\rm NL}$',
          title='reduced skewness')

ax[1].loglog(M_array, sigma_M)
ax[1].set(xlabel=r'$M\ [M_\odot]$', ylabel=r'$\sigma(M)$', title='variance, $z=0$')
for a in ax: a.grid(alpha=.3)
fig.tight_layout()

### Check: $\kappa_3$ is redshift independent

In linear theory $\sigma^2 \propto D(z)^2$ and $\mu_3 \propto D(z)^3$, so
$\kappa_3 = \mu_3/\sigma^3$ does not depend on redshift. The simulation relies
on this: it builds the tables once at $z = 0$ and reuses them at every step,
which is the difference between one expensive calculation and one per redshift.

The check below is cheap and catches a whole class of mistakes — most usefully,
a $\mu_3$ table built at a different redshift from the $\sigma$ it is divided by,
which would give $\kappa_3$ a spurious $D^3$.

In [ ]:
D = lambda z: CLASS_OUTPUT.scale_independent_growth_factor(z)

for z in (0., 5., 10., 20.):
    # mu3 scales as D^3 and sigma as D, so both are scaled consistently here
    k3_z = (np.diag(mu3) * D(z)**3) / (sigma_M * D(z))**3
    print(f"  z = {z:5.1f}   D = {D(z):.4f}   "
          f"kappa3(1e10 Msun) = {np.interp(1e10, M_array, k3_z):.6e}")

print("\nAll four values must agree: kappa_3 carries no redshift dependence.")

## 5. The Edgeworth correction to the mass function

The correction multiplying the Gaussian mass function is, to third order in the
skewness (Eqs. 38–40 of arXiv:2009.01245),

$$\frac{n'_{\rm NG}}{n'_{\rm G}} = 1 + \frac{F_1'}{F_0'} + \frac{F_2'}{F_0'} + \frac{F_3'}{F_0'}$$

with

$$\frac{F_1'}{F_0'} = \frac{\kappa_3 H_3(\nu)}{6} - \frac{H_2(\nu)}{6} X,\qquad
\frac{F_2'}{F_0'} = \frac{\kappa_3^2 H_6}{72} - \frac{\kappa_3 H_5}{36} X,\qquad
\frac{F_3'}{F_0'} = \frac{\kappa_3^3 H_9}{1296} - \frac{\kappa_3^2 H_8}{432} X$$

where $H_n$ are the probabilists' Hermite polynomials, $\nu = B/\sigma$ is the
peak height, and

$$X \equiv \frac{{\rm d}\kappa_3/{\rm d}M}{{\rm d}\nu/{\rm d}M}
      = \frac{1}{\nu}\left[3\kappa_3 - \frac{2 S\, \mu_3'}{\sigma^3 S'}\right].$$

**$X$ deserves attention.** The two terms in that bracket nearly cancel, leaving
a small negative residual of order $-0.4\,\kappa_3$. Relative errors in
$\mu_3'$ or $S'$ are therefore amplified by roughly an order of magnitude, and a
sign error anywhere is catastrophic rather than subtle. A useful sanity check is
that $X\nu/\kappa_3$ should come out small and negative; if it comes out near
$+3$, the $\mu_3'$ term is being lost.

**Barrier.** The code uses $B = \sqrt{a}\,\delta_c$ with `SHETH_a` $= 0.73$, the
21cmFAST value from Jenkins et al. 2001. Note this is *not* the $0.707$ of
Eq. (19) of arXiv:2009.01245, so the barrier here is $1.44$ rather than $1.42$.
The difference is about 1% in $\nu$ and a few per cent in the correction.

In [ ]:
SHETH_a = 0.73      # Constants.h; the paper uses 0.707
DELTAC  = 1.68


def edgeworth_ratio(sigma, dSdM, mu3_diag, dmu3_dM, growthf, return_orders=False):
    '''Non-Gaussian correction to the unconditional halo mass function.

    Mirrors dNion_General() in src/ps.c. Returns n'_NG / n'_G.
    '''
    S, sigma3 = sigma**2, sigma**3
    nu = np.sqrt(SHETH_a) * DELTAC / growthf / sigma
    kappa3 = mu3_diag / sigma3

    X = (3.0*kappa3 - 2.0*S*dmu3_dM/(sigma3*dSdM)) / nu

    H2 = nu**2 - 1.
    H3 = nu**3 - 3*nu
    H5 = nu**5 - 10*nu**3 + 15*nu
    H6 = nu**6 - 15*nu**4 + 45*nu**2 - 15
    H8 = nu**8 - 28*nu**6 + 210*nu**4 - 420*nu**2 + 105
    H9 = nu**9 - 36*nu**7 + 378*nu**5 - 1260*nu**3 + 945*nu

    r1 = kappa3*H3/6. - H2/6.*X
    r2 = kappa3**2*H6/72. - kappa3*H5/36.*X
    r3 = kappa3**3*H9/1296. - kappa3**2*H8/432.*X

    ratio = np.where(np.isfinite(1.+r1+r2+r3), 1.+r1+r2+r3, 1.)
    ratio = np.maximum(ratio, 0.)      # clipped at zero, as the C code does
    return (ratio, r1, r2, r3, nu, X) if return_orders else ratio


# Derivatives on the same grid the tables use
dSdM = np.gradient(sigma_M**2, M_array)
dmu3_dM = np.diag(dmu3_diag)

ratio, r1, r2, r3, nu, X = edgeworth_ratio(
    sigma_M, dSdM, np.diag(mu3), dmu3_dM, D(10.), return_orders=True)

sel = (M_array > 1e7) & (M_array < 1e14)
print(f"X*nu/kappa3 over 1e7-1e14 Msun: "
      f"{(X*nu/kappa3)[sel].min():+.3f} .. {(X*nu/kappa3)[sel].max():+.3f}")
print("  (expect small and negative; near +3 would mean the mu3' term is lost)")

### Convergence of the series

The Edgeworth expansion is asymptotic, not convergent. Successive terms must
decrease; where they do not, the result at that mass is not meaningful however
smooth the curve looks. The second panel below is the one to read before
trusting any of the others.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].semilogx(M_array[sel], ratio[sel], 'k', lw=2, label='total')
ax[0].semilogx(M_array[sel], (1+r1)[sel], '--', label='1st order')
ax[0].semilogx(M_array[sel], (1+r1+r2)[sel], ':', label='1st + 2nd')
ax[0].axhline(1., color='0.6', lw=.8)
ax[0].set(xlabel=r'$M\ [M_\odot]$', ylabel=r"$n'_{\rm NG}/n'_{\rm G}$",
          title=f'correction at $z=10$, $f_{{\\rm NL}}={cosmo_params.F_NL:.0f}$')
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].loglog(M_array[sel], np.abs(r1[sel]), label='$|F_1\'/F_0\'|$')
ax[1].loglog(M_array[sel], np.abs(r2[sel]), label='$|F_2\'/F_0\'|$')
ax[1].loglog(M_array[sel], np.abs(r3[sel]), label='$|F_3\'/F_0\'|$')
ax[1].set(xlabel=r'$M\ [M_\odot]$', ylabel='magnitude of each order',
          title='series must decrease order by order')
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout()

bad = M_array[sel][np.abs(r3[sel]) > np.abs(r2[sel])]
if len(bad):
    print(f"Series stops decreasing above M = {bad.min():.2e} Msun: "
          "the expansion has broken down there.")
else:
    print("Series decreases order by order across the whole range shown.")

## 6. Dependence on $f_{\rm NL}$ and on redshift

Two behaviours to note, both of which matter when interpreting a reionization
history:

* The correction grows steeply with $\nu$, so it is largest for rare haloes:
  high mass, high redshift. At the atomic-cooling threshold during the epoch of
  reionization, $\nu \sim 3\!-\!5$.
* For $f_{\rm NL} < 0$ the correction can pass through zero at the high-mass
  end, where it is clipped. Clipping at zero rather than at one keeps the
  correction continuous in $M$, which matters because this quantity is
  integrated over $\mathrm{d}\ln M$ by an adaptive routine. Watch for the curve
  reaching zero inside the mass range that carries your ionizing emissivity: if
  it does, the regularisation is driving the result.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for fnl in (-500., -100., 100., 500.):
    scale = fnl / cosmo_params.F_NL          # mu3 is linear in f_NL
    r = edgeworth_ratio(sigma_M, dSdM, np.diag(mu3)*scale, dmu3_dM*scale, D(10.))
    ax[0].semilogx(M_array[sel], r[sel], label=f'$f_{{\\rm NL}}={fnl:+.0f}$')
ax[0].axhline(1., color='0.6', lw=.8)
ax[0].set(xlabel=r'$M\ [M_\odot]$', ylabel=r"$n'_{\rm NG}/n'_{\rm G}$",
          title='$z = 10$')
ax[0].legend(); ax[0].grid(alpha=.3)

for z in (6., 10., 15., 20.):
    r = edgeworth_ratio(sigma_M, dSdM, np.diag(mu3), dmu3_dM, D(z))
    ax[1].semilogx(M_array[sel], r[sel], label=f'$z={z:.0f}$')
ax[1].axhline(1., color='0.6', lw=.8)
ax[1].set(xlabel=r'$M\ [M_\odot]$', ylabel=r"$n'_{\rm NG}/n'_{\rm G}$",
          title=f'$f_{{\\rm NL}} = {cosmo_params.F_NL:.0f}$')
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout()

## 7. Dependence on the cut-off scale

$k_{\rm cut}$ decides which modes carry non-Gaussianity. Raising it removes
large-scale power from the bispectrum, lowering $\kappa_3$ and pushing the
turnover in $\kappa_3(M)$ down to smaller masses.

This is what makes small-scale non-Gaussianity interesting: CMB bispectrum
measurements constrain $f_{\rm NL}$ on large scales, but say much less about
modes that only 21-cm observations reach.

Re-running `tpf` for each cut-off is the expensive part, so this cell takes a
few minutes.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))

for kcut in (1e-2, 1e-1, 1.0):
    cp = CosmoParams()
    cp.KCUT_FNL = kcut
    mu3_k, *_ = tpf(log10_M_array, cp, global_params)
    k3_k = np.diag(mu3_k) / sigma_M**3
    Mc = 4*np.pi*rho_m / (3*kcut**3)
    line, = ax.semilogx(M_array, 1e4*k3_k/cp.F_NL,
                        label=rf'$k_{{\rm cut}}={kcut:g}$ Mpc$^{{-1}}$')
    ax.axvline(Mc, ls=':', lw=.8, color=line.get_color())

ax.set(xlabel=r'$M\ [M_\odot]$', ylabel=r'$10^4\,\kappa_3/f_{\rm NL}$',
       title='dotted lines mark $M_{\\rm cut} = 4\\pi\\rho_m/3k_{\\rm cut}^3$')
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()

## Summary

The non-Gaussian correction to the halo mass function is controlled by two
numbers: the reduced skewness $\kappa_3 \propto f_{\rm NL}$, and the peak height
$\nu$, which grows with mass and with redshift. The correction is largest
exactly where haloes are rarest, which is why the epoch of reionization is a
useful place to look for it.

Things worth re-checking whenever this calculation is modified:

| Check | What it catches |
|---|---|
| $\sigma_8$ against Planck | an overall normalisation error, which $\kappa_3$ cannot see |
| $\kappa_3$ equal at two redshifts | a $\mu_3$ table built at the wrong redshift |
| $\kappa_3$ amplitude against Fig. 4 of arXiv:2009.01245 | a wrong $f_{\rm NL}$ convention (a factor 3/5 or 1.3) |
| turnover at $M_{\rm cut} = 4\pi\rho_m/3k_{\rm cut}^3$ | the cut-off not being applied to every leg |
| $X\nu/\kappa_3$ small and negative | a lost or sign-flipped $\mu_3'$ term |
| each order smaller than the last | the asymptotic series having broken down |

The next notebook, `notebook_fNL_simulations.ipynb`, carries these corrections
through to the 21-cm signal.